In [1]:
from src.data import Dataloader
import pandas as pd
import os
from src.utils import seed_everything
import random
from tqdm import tqdm
from typing import Callable
import torch.nn.functional as F
from src.data import TimeSeriesDataset
import torch
from src.nn import MLP, SlidingWindowBinaryClassification
from sklearn.metrics import roc_auc_score, precision_score


## 1. Set up
To set up experience environment, we perform the following steps:

1. This section below will set up `const` needed for experiment
2. For reproducibility, we also set the same inital seed for everything (`numpy`, `torch`)

In [ ]:
# Step 1.
DATA_PATH = os.path.abspath('data/clean/')
VAL_START_DATE = int(pd.Timestamp('2023-12-20').timestamp())  # Unix time in seconds
TEST_START_DATE = int(pd.Timestamp('2024-12-20').timestamp()) # Unix time in seconds
FEAT_COLUMN = ['Close','High','Low', 'Open']
BINARY_LABEL_COLUMN = 'price_increase'
REGRESSION_LABEL_COLUMN = 'next_close'

INIT_SEED = 720

# Step 2.
seed_everything(INIT_SEED)

## 2. Prepare dataset for training

1. Shuffle all token networks (I.I.D)
2. Split datasets for training, validation and testing: the first 70% of datasets for training, the next 15% for validation, the next 15% for testing
3. Load all dataset in memory to `TimeSeriesDataset`
4. For each token, we use all data before `2023-12-20` for trainning, from `2023-12-20` to `2024-12-20` for validation and `2024-12-20` onward for testing

There are 49 tokens datasets in total, so we keep 33 networks for training, 16 tokens for validation and 16 tokens for testing.

In [13]:
files = os.listdir(DATA_PATH)
random.shuffle(files) # Step 1

# Step 2
train_token_list = files[:33]
valid_token_list = files[33:33+8]
test_token_list = files [-8:]

assert len(set(train_token_list).intersection(set(valid_token_list))) == 0
assert len(set(test_token_list).intersection(set(valid_token_list))) == 0

# Step 3
data_loader = Dataloader(DATA_PATH)

train_data = []
valid_data = []
test_data = []

for file_name in tqdm(train_token_list):
    train_data.append(data_loader.from_csv(file_name, feat_columns=FEAT_COLUMN, label_column=BINARY_LABEL_COLUMN))

for file_name in tqdm(valid_token_list):
    valid_data.append(data_loader.from_csv(file_name, feat_columns=FEAT_COLUMN, label_column=BINARY_LABEL_COLUMN))

for file_name in tqdm(test_token_list):
    test_data.append(data_loader.from_csv(file_name, feat_columns=FEAT_COLUMN, label_column=BINARY_LABEL_COLUMN))

train_data_split = []
valid_data_split = []
test_data_split = []

for data in tqdm(train_data):
    train, val_test = data.split(VAL_START_DATE)
    val, test = val_test.split(TEST_START_DATE)
    train_data_split.append((train,val,test))

for data in tqdm(valid_data):
    train, val_test = data.split(VAL_START_DATE)
    val, test = val_test.split(TEST_START_DATE)
    valid_data_split.append((train,val,test))

for data in tqdm(test_data):
    train, val_test = data.split(VAL_START_DATE)
    val, test = val_test.split(TEST_START_DATE)
    test_data_split.append((train,val,test))

100%|██████████| 8/8 [00:00<00:00, 1995.03it/s]


4. Next we normalize each feature with min and max of each feature

In [14]:
def normalize(train_data: TimeSeriesDataset, valid_data: TimeSeriesDataset, test_data: TimeSeriesDataset, normalize_y = False):
    
    max_values,_ = torch.max(train_data.x, dim=0)
    min_values,_ = torch.min(train_data.x, dim=0)

    train_data.x = (train_data.x - min_values) / (max_values - min_values)
    valid_data.x = (valid_data.x - min_values) / (max_values - min_values)
    test_data.x = (test_data.x - min_values) / (max_values - min_values)

    if normalize_y:
        max_values,_ = torch.max(train_data.y, dim=0)
        min_values,_ = torch.min(train_data.y, dim=0)

        train_data.y = (train_data.y - min_values) / (max_values - min_values)
        valid_data.y = (valid_data.y - min_values) / (max_values - min_values)
        test_data.y = (test_data.y - min_values) / (max_values - min_values)

    return train_data,valid_data, test_data


for data_split in [train_data_split, valid_data_split, test_data_split]:
    for train,val, test in data_split:
        normalize(train,val,test)
        


## 3. Traditional machine learning baseline

In this project, we explore the following traditional machine learning for binary classification
1. Logistic Regression
2. Random Forest
3. Gradient Boosting (GBM)
4. SVM
5. KNN

In [27]:
train_0, val_0, test_0 = train_data_split[0]
train_1, val_1, test_1 = train_data_split[1]

print(train_0.x.shape)
print(train_1.x.shape)
torch.concatenate([train_0.x,train_1.x]).shape



torch.Size([799, 4])
torch.Size([774, 4])


torch.Size([1573, 4])

In [32]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score
import numpy as np

def traditional_model_experience(n_tokens, train_data, test_data):
    used_train_data = train_data[:n_tokens]
    all_train_x = []
    all_train_y = []
    all_test_x = []
    all_test_y = []

    for train, _, test in used_train_data:
        all_train_x.append(train.x)
        all_train_y.append(train.y)
        
    for train, _, test in test_data: 
        all_test_y.append(test.y)
        all_test_x.append(test.x)

    train_x = torch.concatenate(all_train_x).numpy()
    train_y = torch.concatenate(all_train_y).numpy()
    test_x = torch.concatenate(all_test_x).numpy()
    test_y = torch.concatenate(all_test_y).squeeze().numpy()

    models = {
        "Logistic Regression":     LogisticRegression(),
        "Random Forest":           RandomForestClassifier(),
        "Gradient Boosting (GBM)": GradientBoostingClassifier(),
        "SVM":                     SVC(probability=True),
        "KNN":                     KNeighborsClassifier(),
    }
    print(np.isnan(test_x).any())
    for name, model in models.items():
        model.fit(train_x, train_y)
        preds = model.predict_proba(test_x)[:, 1]
        auc = roc_auc_score(test_y, preds)
        print(f"{name:30s} AUC: {auc:.4f}")



# train_data,_,test_data = train_data_split[1]
# X_train, y_train = train_data.x.numpy(), train_data.y.squeeze().numpy()
# X_test,  y_test  = test_data.x.numpy(),  test_data.y.squeeze().numpy()

# models = {
#     "Logistic Regression":     LogisticRegression(),
#     "Random Forest":           RandomForestClassifier(),
#     "Gradient Boosting (GBM)": GradientBoostingClassifier(),
#     "SVM":                     SVC(probability=True),
#     "KNN":                     KNeighborsClassifier(),
# }

# for name, clf in models.items():
#     clf.fit(X_train, y_train)
#     preds = clf.predict_proba(X_test)[:, 1]
#     auc = roc_auc_score(y_test, preds)
#     print(f"{name:30s} AUC: {auc:.4f}")

traditional_model_experience(1,train_data_split,test_data_split )


True


d:\anaconda\envs\data2010\lib\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

## Training loop

In [ ]:

data_loader = Dataloader(DATA_PATH)
timeseries_data = data_loader.from_csv("aave.csv", feat_columns = FEAT_COLUMN, label_column=BINARY_LABEL_COLUMN)

train_data, val_test_data = timeseries_data.split(VAL_START_DATE)
val_data, test_data = val_test_data.split(TEST_START_DATE)


train_data, val_data, test_data = normalize(train_data,val_data, test_data)
print(train_data.x.shape)


def train(train_data : TimeSeriesDataset, val_data : TimeSeriesDataset, model: torch.nn.Module, criterion : Callable, opt: torch.optim.Optimizer, epoch = 50):
    model.train()
    for i in range(epoch):
        all_loss = []
        preds = []
        for time, feat, y in train_data:
            opt.zero_grad()
            z = model(feat.float())
            loss = criterion(
                z.float(),y.float()
            )
            
            all_loss.append(loss.item())
            loss.backward()
            opt.step()

            preds.append(z.sigmoid().item())
        auc = roc_auc_score(train_data.y.squeeze().tolist(), preds)
        print(f"[INFO] Epoch - {i+1} : Loss - {sum(all_loss)/float(len(all_loss))} ; AUC : {auc}")

def evaluate(data: TimeSeriesDataset, model: torch.nn.Module, evaluator: Callable = roc_auc_score):
    model.eval()
    preds = []
    for time, feat, y in data:
        pred = model(feat).sigmoid()
        
        preds.append(pred.item())
    return evaluator(data.y.squeeze().tolist(), preds)


model = MLP(in_channel= 4, out_channel= 1, dim = 128, num_layers=3)

opt = torch.optim.Adam(
    model.parameters(), lr=float(0.001)
)

score = evaluate(test_data,model)
print(score)

train(train_data, val_data, model, F.binary_cross_entropy_with_logits,opt)

score = evaluate(test_data,model)
print(score)

           


0.5542926506290002
[INFO] Epoch - 1 : Loss - 0.6960590809993958 ; AUC : 0.4786278195488721
[INFO] Epoch - 2 : Loss - 0.6942884749703772 ; AUC : 0.49703634085213033
[INFO] Epoch - 3 : Loss - 0.6936243853073693 ; AUC : 0.506218671679198
[INFO] Epoch - 4 : Loss - 0.6930096171078306 ; AUC : 0.5154323308270677
[INFO] Epoch - 5 : Loss - 0.6923198456012263 ; AUC : 0.5216917293233083
[INFO] Epoch - 6 : Loss - 0.6917311185143319 ; AUC : 0.5272243107769423
[INFO] Epoch - 7 : Loss - 0.6916595277410276 ; AUC : 0.5281892230576442
[INFO] Epoch - 8 : Loss - 0.6915778748980154 ; AUC : 0.5275971177944863
[INFO] Epoch - 9 : Loss - 0.691253974642115 ; AUC : 0.530921052631579
[INFO] Epoch - 10 : Loss - 0.6911088466942683 ; AUC : 0.5334273182957394
[INFO] Epoch - 11 : Loss - 0.6910104204775842 ; AUC : 0.5333333333333334
[INFO] Epoch - 12 : Loss - 0.6909934980923005 ; AUC : 0.5336591478696743
[INFO] Epoch - 13 : Loss - 0.690792130252745 ; AUC : 0.5334649122807017
[INFO] Epoch - 14 : Loss - 0.690731578237273

In [11]:
# 1. Check label balance
labels = train_data.y.squeeze().tolist()
print(f"Positive rate: {sum(labels)/len(labels):.4f}")  # should be near 0.5

# 2. Check model output variance BEFORE training
model.eval()
outs = []
with torch.no_grad():
   for time, feat, y  in train_data:
        z = model(feat.float()).sigmoid().item()
        outs.append(z)

import numpy as np
print(f"Output mean: {np.mean(outs):.4f}")
print(f"Output std:  {np.std(outs):.4f}")  # near 0 = model outputs same thing for all inputs

# 3. Check feature variance after normalization
print(f"Feature mean: {train_data.x.mean(dim=0)}")
print(f"Feature std:  {train_data.x.std(dim=0)}")  # any column near 0 = dead feature

# 4. Check sizes
print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")
print(f"Train labels: {train_data.y.shape}")
print(f"Train features: {train_data.x.shape}")


Positive rate: 0.5006
Output mean: 0.5111
Output std:  0.0685
Feature mean: tensor([0.2086, 0.1568, 0.2130, 0.2095])
Feature std:  tensor([0.2333, 0.1765, 0.2308, 0.2344])
Train: 799, Val: 366, Test: 381
Train labels: torch.Size([799, 1])
Train features: torch.Size([799, 4])


## Sliding window baseline

In [ ]:
baseline = SlidingWindowBinaryClassification()

auc = []
for data in test_data_split:
    train, val, test = data
    preds = []
    for _, _, y in val:
        baseline.update(y.item())
    
    for _,_,y in test:
        y = baseline()
        preds.append(y)
        baseline.update(y) 

    score = roc_auc_score(test.y.squeeze().tolist(), preds)
    auc.append(score)

print(sum(auc)/len(auc))
    
    

torch.Size([381, 1])
381
torch.Size([381, 1])
381
torch.Size([381, 1])
381
torch.Size([381, 1])
381
torch.Size([381, 1])
381
torch.Size([381, 1])
381
torch.Size([381, 1])
381
torch.Size([381, 1])
381
0.5
